# geoarrow_types

## Example 1: each geometry type has its GeoArrow extension name

In [ ]:
# Example 1: wkb/wkt and the six native geometry types map to their geoarrow.* extension names
import geoarrow.types as gt

names = {f.__name__: f().extension_name() for f in (gt.wkb, gt.wkt, gt.point, gt.linestring, gt.polygon,
                                                     gt.multipoint, gt.multilinestring, gt.multipolygon)}
assert names["wkb"] == "geoarrow.wkb" and names["point"] == "geoarrow.point"
assert names["multipolygon"] == "geoarrow.multipolygon" and gt.large_wkb().extension_name() == "geoarrow.wkb"
print("Example 1:", names)

## Example 2: dimensions, coordinate types and encodings

In [ ]:
# Example 2: a TypeSpec carries encoding, geometry type, dimensions and coordinate type
import geoarrow.types as gt

spec = gt.point(dimensions=gt.Dimensions.XYZ, coord_type=gt.CoordType.INTERLEAVED)
assert spec.encoding == gt.Encoding.GEOARROW and spec.geometry_type == gt.GeometryType.POINT
assert spec.dimensions == gt.Dimensions.XYZ and spec.coord_type == gt.CoordType.INTERLEAVED
assert repr(gt.TypeSpec()) == "TypeSpec()" and repr(gt.TypeSpec(encoding=gt.Encoding.WKB)) == "TypeSpec(Encoding.WKB)"
print("Example 2:", spec)

## Example 3: CRS and edge metadata

In [ ]:
# Example 3: the extension metadata holds the CRS (PROJJSON) and the edge interpretation
import json
import geoarrow.types as gt

assert gt.TypeSpec().with_defaults().extension_metadata() == "{}"
spherical = gt.TypeSpec(edge_type=gt.EdgeType.SPHERICAL).with_defaults().extension_metadata()
assert spherical == '{"edges": "spherical"}', spherical
with_crs = json.loads(gt.TypeSpec(crs=gt.OGC_CRS84).with_defaults().extension_metadata())
assert repr(gt.OGC_CRS84) == "ProjJsonCrs(OGC:CRS84)" and with_crs["crs"]["id"]["code"] == "CRS84"
print("Example 3:", spherical, "| crs id:", with_crs["crs"]["id"])

## Example 4: the pyarrow extension types

In [ ]:
# Example 4: to_pyarrow() builds the registered pyarrow extension type for a spec
import pyarrow as pa
import geoarrow.types as gt
from geoarrow.types import type_pyarrow

wkb = gt.wkb().to_pyarrow()
assert isinstance(wkb, type_pyarrow.WkbType) and wkb.encoding == gt.Encoding.WKB
assert wkb.storage_type == pa.binary() and wkb.extension_name == "geoarrow.wkb"
point = gt.point().to_pyarrow()
assert point.storage_type.names == ["x", "y"]
print("Example 4:", wkb, "|", point.storage_type)

## Example 5: wrap a struct array as a point array

In [ ]:
# Example 5: wrap_array turns an x/y struct array into a geoarrow.point extension array
import pyarrow as pa
from pyarrow import compute as pc
import geoarrow.types as gt

storage = pc.make_struct(pa.array([1.0, 2.0, 3.0]), pa.array([3.0, 4.0, 5.0]), field_names=["x", "y"])
points = gt.point().to_pyarrow().wrap_array(storage)
assert len(points) == 3 and points.type.extension_name == "geoarrow.point"
assert points.storage.field("x").to_pylist() == [1.0, 2.0, 3.0]
print("Example 5:", points.type, points.storage.to_pylist()[:2])